<p style="text-align:center">
    <a href="https://skills.network" target="_blank">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="200" alt="Skills Network Logo">
    </a>
</p>


# **Launch Sites Locations Analysis with Folium**


Estimated time needed: **40** minutes


The launch success rate may depend on many factors such as payload mass, orbit type, and so on. It may also depend on the location and proximities of a launch site, i.e., the initial position of rocket trajectories. Finding an optimal location for building a launch site certainly involves many factors and hopefully we could discover some of the factors by analyzing the existing launch site locations.


In the previous exploratory data analysis labs, you have visualized the SpaceX launch dataset using `matplotlib` and `seaborn` and discovered some preliminary correlations between the launch site and success rates. In this lab, you will be performing more interactive visual analytics using `Folium`.


## Objectives


This lab contains the following tasks:
- **TASK 1:** Mark all launch sites on a map
- **TASK 2:** Mark the success/failed launches for each site on the map
- **TASK 3:** Calculate the distances between a launch site to its proximities

After completed the above tasks, you should be able to find some geographical patterns about launch sites.


Let's first import required Python packages for this lab:


In [1]:
# Install only missing dependencies into the notebook's current Python.
import importlib.util
import subprocess
import sys
missing = [name for name in ("folium", "pandas")
           if importlib.util.find_spec(name) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])

In [2]:
import folium
import pandas as pd
from pathlib import Path
from html import escape

LAB_DIR = Path("capstone") if Path("capstone").is_dir() else Path(".")
DATA_DIR = LAB_DIR / "data"
OUTPUT_DIR = LAB_DIR / "location_outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

/Users/vavanfarizzaman/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [3]:
from folium.plugins import MarkerCluster, MousePosition
from folium.features import DivIcon


def map_heading(map_object, title, subtitle):
    map_object.get_root().html.add_child(folium.Element(
        '<div style="position:fixed;top:12px;left:60px;z-index:999;'
        'background:white;padding:12px 16px;border-radius:6px;'
        'box-shadow:0 1px 6px #999;font:14px Arial;max-width:340px">'
        f'<b>{escape(title)}</b><br><span style="font-size:12px">'
        f'{escape(subtitle)}</span></div>'
    ))


def make_site_map():
    result = folium.Map(location=[31, -99], zoom_start=4, control_scale=True)
    site_layer = folium.FeatureGroup(name="Course launch-site coordinates").add_to(result)
    for _, row in launch_sites_df.iterrows():
        coordinate = [row["Lat"], row["Long"]]
        name = row["Launch Site"]
        folium.Circle(
            coordinate, radius=1000, color="#0072B2", fill=True, fill_opacity=0.15,
            tooltip=name, popup=folium.Popup(escape(name), max_width=250),
        ).add_to(site_layer)
        label_offset = -25 if name == "CCAFS SLC-40" else 5
        folium.Marker(
            coordinate,
            icon=DivIcon(
                icon_size=(150, 24), icon_anchor=(-12, label_offset),
                html='<div style="white-space:nowrap;background:white;border:1px solid #0072B2;'
                     'border-radius:4px;padding:3px 6px;font:12px Arial;color:#005580">'
                     f'{escape(name)}</div>',
            ),
            tooltip=name,
        ).add_to(site_layer)
    result.fit_bounds(launch_sites_df[["Lat", "Long"]].values.tolist(), padding=(45, 45))
    return result

If you need to refresh your memory about folium, you may download and refer to this previous folium lab:


[Generating Maps with Python](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/DV0101EN-3-5-1-Generating-Maps-in-Python-py-v2.0.ipynb)


## Task 1: Mark all launch sites on a map


First, let's try to add each site's location on a map using site's latitude and longitude coordinates


Use the supplied local `data/spacex_launch_geo.csv`. This mapping lab has 56 launches and four site labels, a different subset from the earlier API and SQL labs. Course coordinates are preserved as supplied. Run the cells in order; three interactive maps and a distance table are saved in `location_outputs/`.

The data and calculations run locally. Map backgrounds and Folium's JavaScript/CSS are loaded from online services when you view a map. Open the exported HTML files in a browser if your notebook viewer blocks interactive output.

In [4]:
spacex_csv_file = DATA_DIR / "spacex_launch_geo.csv"
if not spacex_csv_file.is_file():
    raise FileNotFoundError(f"Missing supplied dataset: {spacex_csv_file}")
raw_spacex_df = pd.read_csv(spacex_csv_file)
required_columns = {"Launch Site", "Lat", "Long", "class"}
if required_columns - set(raw_spacex_df):
    raise ValueError(f"Missing columns: {sorted(required_columns - set(raw_spacex_df))}")
if raw_spacex_df.empty or not raw_spacex_df["class"].isin([0, 1]).all():
    raise ValueError("Expected a nonempty dataset with binary class labels.")
if not raw_spacex_df["Lat"].between(-90, 90).all() or not raw_spacex_df["Long"].between(-180, 180).all():
    raise ValueError("Invalid or missing launch coordinates.")
print(f"Loaded {len(raw_spacex_df)} launch records.")

Loaded 56 launch records.


Now, you can take a look at what are the coordinates for each site.


In [5]:
spacex_df = raw_spacex_df[["Launch Site", "Lat", "Long", "class"]].copy()
if (spacex_df.groupby("Launch Site")[["Lat", "Long"]].nunique() > 1).any().any():
    raise ValueError("A launch-site label has inconsistent coordinates.")
launch_sites_df = spacex_df[["Launch Site", "Lat", "Long"]].drop_duplicates().reset_index(drop=True)
launch_sites_df

,Launch Site,Lat,Long
0,CCAFS LC-40,28.562302,-80.577356
1,VAFB SLC-4E,34.632834,-120.610745
2,KSC LC-39A,28.573255,-80.646895
3,CCAFS SLC-40,28.563197,-80.576820


Above coordinates are just plain numbers that can not give you any intuitive insights about where are those launch sites. If you are very good at geography, you can interpret those numbers directly in your mind. If not, that's fine too. Let's visualize those locations by pinning them on a map.


We first need to create a folium `Map` object, with an initial center location to be NASA Johnson Space Center at Houston, Texas.


In [6]:
nasa_coordinate = [29.559684888503615, -95.0830971930759]
nasa_map = folium.Map(location=nasa_coordinate, zoom_start=10, control_scale=True)

We could use `folium.Circle` to add a highlighted circle area with a text label on a specific coordinate. For example, 


In [7]:
# Demonstration marker on a separate map; never reused as a launch marker.
folium.Circle(
    nasa_coordinate, radius=1000, color="#d35400", fill=True,
    popup="NASA Johnson Space Center",
).add_to(nasa_map)
folium.Marker(
    nasa_coordinate,
    icon=DivIcon(
        icon_size=(130, 24), icon_anchor=(0, 0),
        html='<div style="font-size:12px;color:#d35400;white-space:nowrap"><b>NASA JSC</b></div>',
    ),
).add_to(nasa_map)
nasa_map

The orange circle marks NASA Johnson Space Center. Zoom in to inspect the demonstration; the launch sites are added to their own map below.

Now, let's add a circle for each launch site in data frame `launch_sites`


Create a circle and a named marker for each distinct launch-site label.

An example of folium.Circle:


`folium.Circle(coordinate, radius=1000, color='#000000', fill=True).add_child(folium.Popup(...))`


An example of folium.Marker:


`folium.map.Marker(coordinate, icon=DivIcon(icon_size=(20,20),icon_anchor=(0,0), html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % 'label', ))`


In [8]:
site_map = make_site_map()
map_heading(site_map, "SpaceX launch-site locations", "Four site labels from the course dataset")
folium.LayerControl().add_to(site_map)
site_map.save(OUTPUT_DIR / "01_launch_sites.html")
site_map

The generated map with marked launch sites should look similar to the following:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/launch_site_markers.png">
</center>


**Findings:** The supplied latitudes range from about 28.6°N to 34.6°N. These sites are not on or immediately next to the Equator; they are in the southern part of the continental United States. They are in coastal regions of Florida and California. Zoom in to inspect each location. The two Cape Canaveral labels are plotted separately as in the source dataset and sit very close together.

# Task 2: Mark the success/failed launches for each site on the map


Next, let's try to enhance the map by adding the launch outcomes for each site, and see which sites have high success rates.
Recall that data frame spacex_df has detailed launch records, and the `class` column indicates if this launch was successful or not


In [9]:
spacex_df.tail(10)

,Launch Site,Lat,Long,class
46,KSC LC-39A,28.573255,-80.646895,1
47,KSC LC-39A,28.573255,-80.646895,1
48,KSC LC-39A,28.573255,-80.646895,1
49,CCAFS SLC-40,28.563197,-80.576820,1
50,CCAFS SLC-40,28.563197,-80.576820,1
51,CCAFS SLC-40,28.563197,-80.576820,0
52,CCAFS SLC-40,28.563197,-80.576820,0
53,CCAFS SLC-40,28.563197,-80.576820,0
54,CCAFS SLC-40,28.563197,-80.576820,1
55,CCAFS SLC-40,28.563197,-80.576820,0


Next, let's create markers for all launch records. 
If a launch was successful `(class=1)`, then we use a green marker and if a launch was failed, we use a red marker `(class=0)`


Note that a launch only happens in one of the four launch sites, which means many launch records will have the exact same coordinate. Marker clusters can be a good way to simplify a map containing many markers having the same coordinate.


Let's first create a `MarkerCluster` object


In [10]:
marker_cluster = MarkerCluster(name="Individual launches", spiderfy_on_max_zoom=True)

Create `marker_color` on the per-launch `spacex_df`, not on the deduplicated site table: green for class 1 and red for class 0.

In [11]:
def assign_marker_color(launch_outcome):
    if launch_outcome not in (0, 1):
        raise ValueError(f"Unexpected launch class: {launch_outcome!r}")
    return {0: "red", 1: "green"}[launch_outcome]

In [12]:
spacex_df["marker_color"] = spacex_df["class"].apply(assign_marker_color)
spacex_df.tail(10)

,Launch Site,Lat,Long,class,marker_color
46,KSC LC-39A,28.573255,-80.646895,1,green
47,KSC LC-39A,28.573255,-80.646895,1,green
48,KSC LC-39A,28.573255,-80.646895,1,green
49,CCAFS SLC-40,28.563197,-80.576820,1,green
50,CCAFS SLC-40,28.563197,-80.576820,1,green
51,CCAFS SLC-40,28.563197,-80.576820,0,red
52,CCAFS SLC-40,28.563197,-80.576820,0,red
53,CCAFS SLC-40,28.563197,-80.576820,0,red
54,CCAFS SLC-40,28.563197,-80.576820,1,green
55,CCAFS SLC-40,28.563197,-80.576820,0,red


Create a separate marker for every launch and add it to a fresh cluster. Clicking a cluster zooms in; overlapping markers spread out at maximum zoom.

In [13]:
# Rebuild on rerun so markers never accumulate.
site_map = make_site_map()
marker_cluster = MarkerCluster(
    name="Individual launches", spiderfy_on_max_zoom=True, show_coverage_on_hover=False,
).add_to(site_map)
for index, record in spacex_df.iterrows():
    outcome = "Successful" if record["class"] == 1 else "Unsuccessful"
    detail = raw_spacex_df.loc[index]
    popup = (
        f"<b>{escape(record['Launch Site'])}</b><br>"
        f"Flight: {escape(str(detail['Flight Number']))}<br>"
        f"Date: {escape(str(detail['Date']))}<br>Outcome: {outcome}"
    )
    folium.Marker(
        [record["Lat"], record["Long"]],
        tooltip=f"{record['Launch Site']} — {outcome}",
        popup=folium.Popup(popup, max_width=280),
        icon=folium.Icon(color=record["marker_color"], icon="ok" if record["class"] == 1 else "remove"),
    ).add_to(marker_cluster)
map_heading(site_map, "Launch outcomes by site",
            f"{len(spacex_df)} launches · Green = class 1 · Red = class 0 · Click clusters to explore")
folium.LayerControl().add_to(site_map)
site_map.save(OUTPUT_DIR / "02_launch_outcomes.html")
site_map

Your updated map may look like the following screenshots:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/launch_site_marker_cluster.png">
</center>


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/launch_site_marker_cluster_zoomed.png">
</center>


**Findings:** In this 56-launch subset, KSC LC-39A has 10 successful launches out of 13 (76.9%); CCAFS SLC-40 has 3 of 7 (42.9%); VAFB SLC-4E has 4 of 10 (40.0%); and CCAFS LC-40 has 7 of 26 (26.9%). These descriptive rates do not establish that location causes a difference in performance; launch periods and mission types also differ.

# TASK 3: Calculate the distances between a launch site to its proximities


Next, we need to explore and analyze the proximities of launch sites.


Let's first add a `MousePosition` on the map to get coordinate for a mouse over a point on the map. As such, while you are exploring the map, you can easily find the coordinates of any points of interests (such as railway)


In [14]:
selected_site = launch_sites_df.loc[launch_sites_df["Launch Site"] == "CCAFS LC-40"].iloc[0]
launch_site_lat, launch_site_lon = float(selected_site["Lat"]), float(selected_site["Long"])
launch_coordinate = [launch_site_lat, launch_site_lon]
proximity_map = folium.Map(location=launch_coordinate, zoom_start=14, control_scale=True)
folium.Marker(launch_coordinate, tooltip="CCAFS LC-40 (course coordinate)",
              popup="Reference launch site: CCAFS LC-40",
              icon=folium.Icon(color="blue", icon="flag")).add_to(proximity_map)
formatter = "function(num) {return L.Util.formatNum(num, 5);};"
mouse_position = MousePosition(
    position="topright", separator=" Long: ", empty_string="Move cursor over map",
    lng_first=False, num_digits=5, prefix="Lat:",
    lat_formatter=formatter, lng_formatter=formatter,
)
proximity_map.add_child(mouse_position)
folium.LatLngPopup().add_to(proximity_map)
map_heading(proximity_map, "CCAFS LC-40: nearby features",
            "Move the cursor for coordinates; click the map to pin a coordinate")
proximity_map

Zoom in and use the **top-right** MousePosition readout to inspect coastline, rail, road, and settlement coordinates. Clicking the map also displays a coordinate popup.

To make the completed calculation reproducible, the selected coordinates below are supplied with OpenStreetMap source links. They were derived from a saved map-data extract rather than guessed or claimed to be manually measured. You can inspect them with MousePosition and replace them to explore other points.

You can calculate the distance between two points on the map based on their `Lat` and `Long` values using the following method:


In [15]:
from math import sin, cos, sqrt, atan2, radians, isfinite


def calculate_distance(lat1, lon1, lat2, lon2):
    """Great-circle distance in kilometres using the lab's 6373 km Earth radius."""
    if not all(isfinite(value) for value in (lat1, lon1, lat2, lon2)):
        raise ValueError("Coordinates must be finite.")
    if not (-90 <= lat1 <= 90 and -90 <= lat2 <= 90 and -180 <= lon1 <= 180 and -180 <= lon2 <= 180):
        raise ValueError("Coordinates are outside latitude/longitude bounds.")
    lat1, lon1, lat2, lon2 = map(radians, (lat1, lon1, lat2, lon2))
    a = sin((lat2 - lat1) / 2) ** 2 + cos(lat1) * cos(lat2) * sin((lon2 - lon1) / 2) ** 2
    a = min(1.0, max(0.0, a))  # Avoid roundoff errors near antipodal points.
    return 6373.0 * 2 * atan2(sqrt(a), sqrt(1 - a))

Select the nearest coastline point from the saved OpenStreetMap extract and calculate its distance from the course's CCAFS LC-40 coordinate. For lines, the selected point lies on a mapped segment, using a local projection approximation. Distances below are straight-line great-circle estimates, not travel distances.

In [16]:
proximity_points = pd.read_csv(DATA_DIR / "launch_site_proximity_points.csv")
proximity_points["Distance_km"] = proximity_points.apply(
    lambda row: calculate_distance(launch_site_lat, launch_site_lon, row["Latitude"], row["Longitude"]),
    axis=1,
)
coastline = proximity_points.loc[proximity_points["Feature"] == "Coastline"].iloc[0]
coastline_lat, coastline_lon = coastline["Latitude"], coastline["Longitude"]
distance_coastline = calculate_distance(launch_site_lat, launch_site_lon, coastline_lat, coastline_lon)
print(f"Coastline distance: {distance_coastline:.3f} km")
# Include the source lines for inspection; enable this layer at high zoom.
import json
osm_snapshot = json.loads((DATA_DIR / "launch_site_osm_features.json").read_text())
selected_sources = set(proximity_points["Source"])
source_layer = folium.FeatureGroup(name="Selected OSM feature geometry", show=False)
for feature in osm_snapshot["data"]["elements"]:
    source = f"https://www.openstreetmap.org/{feature['type']}/{feature['id']}"
    if source in selected_sources and feature["type"] == "way":
        folium.PolyLine(
            [[point["lat"], point["lon"]] for point in feature["geometry"]],
            color="#555555", weight=4, tooltip=feature.get("tags", {}).get("name", source),
        ).add_to(source_layer)
proximity_map.add_child(source_layer, name="source_geometry")
proximity_points[["Feature", "Name", "Latitude", "Longitude", "Distance_km", "Source"]]

Coastline distance: 0.922 km


,Feature,Name,Latitude,Longitude,Distance_km,Source
0,Coastline,Coastline,28.563400,-80.568004,0.921825,https://www.openstreetmap.org/way/22322002
1,Railway,Rail spur,28.562297,-80.577323,0.003293,https://www.openstreetmap.org/way/96332348
2,Major road,Nasa Parkway East,28.515759,-80.615604,6.385049,https://www.openstreetmap.org/way/26948002
3,City/town centre,Cape Canaveral,28.388144,-80.605646,19.568018,https://www.openstreetmap.org/node/154352850


Add a marker for the selected coastline point, with its distance and source.

In [17]:
def feature_marker(record, color):
    coordinate = [record["Latitude"], record["Longitude"]]
    popup = (
        f"<b>{escape(record['Feature'])}: {escape(record['Name'])}</b><br>"
        f"Straight-line distance: {record['Distance_km']:.3f} km<br>"
        f"<a href=\"{escape(record['Source'], quote=True)}\" target=\"_blank\">OpenStreetMap source</a>"
    )
    return folium.Marker(
        coordinate, popup=folium.Popup(popup, max_width=320),
        tooltip=f"{record['Feature']}: {record['Distance_km']:.3f} km",
        icon=DivIcon(
            icon_size=(210, 28), icon_anchor=(0, 0),
            html=f'<div style="white-space:nowrap;background:white;border:2px solid {color};'
                 'border-radius:4px;padding:4px 7px;font:12px Arial">'
                 f'{escape(record["Feature"])}: {record["Distance_km"]:.3f} km</div>',
        ),
    )


coast_layer = folium.FeatureGroup(name="Coastline distance")
distance_marker = feature_marker(coastline, "#0072B2").add_to(coast_layer)
# Stable layer keys make these cells safe to rerun.
proximity_map.add_child(coast_layer, name="coastline_distance")

Draw a line between the launch-site coordinate and the selected coastline point.

In [18]:
lines = folium.PolyLine(
    locations=[launch_coordinate, [coastline_lat, coastline_lon]],
    color="#0072B2", weight=3,
    tooltip=f"Coastline: {distance_coastline:.3f} km (straight line)",
)
coast_layer.add_child(lines, name="coastline_line")
proximity_map

Your updated map with distance line should look like the following screenshot:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/launch_site_marker_distance.png">
</center>


Add distances to a railway, a major road, and a city/town centre. The selected points are nearest **within the saved query's feature classes and search areas**: coastline and rail within 10 km, major roads within 15 km, and city/town nodes within 60 km. Major roads are OSM motorway, trunk, primary, or secondary ways. A town-centre point is not the edge of urban development.

A railway map symbol may look like this:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/railway.png">
</center>


A highway map symbol may look like this:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/highway.png">
</center>


A city map symbol may look like this:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/city.png">
</center>


In [19]:
feature_colors = {"Railway": "#8E44AD", "Major road": "#D55E00", "City/town centre": "#228833"}
nearby_layer = folium.FeatureGroup(name="Rail, road, and settlement distances")
for _, record in proximity_points.loc[proximity_points["Feature"] != "Coastline"].iterrows():
    color = feature_colors[record["Feature"]]
    feature_marker(record, color).add_to(nearby_layer)
    folium.PolyLine(
        locations=[launch_coordinate, [record["Latitude"], record["Longitude"]]],
        color=color, weight=3, dash_array="6 4",
        tooltip=f"{record['Feature']}: {record['Distance_km']:.3f} km (straight line)",
    ).add_to(nearby_layer)
proximity_map.add_child(nearby_layer, name="nearby_distances")
proximity_map.fit_bounds(
    [launch_coordinate] + proximity_points[["Latitude", "Longitude"]].values.tolist(),
    padding=(70, 70),
)
proximity_map.add_child(folium.LayerControl(), name="layer_control")
proximity_map

In [20]:
site_summary = (
    spacex_df.groupby("Launch Site")["class"]
    .agg(Launches="size", Successes="sum", SuccessRate="mean")
)
site_summary

,Launches,Successes,SuccessRate
Launch Site,,,
CCAFS LC-40,26,7,0.269231
CCAFS SLC-40,7,3,0.428571
KSC LC-39A,13,10,0.769231
VAFB SLC-4E,10,4,0.400000


In [21]:
proximity_map.save(OUTPUT_DIR / "03_launch_site_proximity.html")
proximity_points.to_csv(OUTPUT_DIR / "launch_site_distances.csv", index=False)
site_summary.to_csv(OUTPUT_DIR / "launch_site_summary.csv")
print(f"Saved three interactive maps and two CSV tables in {OUTPUT_DIR}")
proximity_points[["Feature", "Name", "Distance_km"]]

Saved three interactive maps and two CSV tables in location_outputs


,Feature,Name,Distance_km
0,Coastline,Coastline,0.921825
1,Railway,Rail spur,0.003293
2,Major road,Nasa Parkway East,6.385049
3,City/town centre,Cape Canaveral,19.568018


**Interpretation:** The selected coastline is about **0.922 km** away, the major road (Nasa Parkway East) **6.385 km**, and Cape Canaveral town centre **19.568 km**. The nearest mapped rail feature is a **spur within the launch complex**, only about **0.003 km** from the course coordinate; this is not a distance to the external rail network. Enable the selected-geometry layer and zoom in to inspect it. This very small number is sensitive to coordinate and map accuracy. These measurements describe the selected CCAFS LC-40 course coordinate; they do not establish nearest distances for every launch site. The coastal setting and nearby transport features are consistent with access to infrastructure and launch paths over water, but this map does not prove why the site was chosen or explain success rates.

All four supplied site labels are in coastal regions and well north of the Equator. The source coordinates are retained for the exercise, and the nearby features reflect the timestamp recorded in the OpenStreetMap extract, not necessarily their configuration at historical launch dates. Sources and the selection query are saved in `data/launch_site_osm_features.json`; `prepare_launch_site_points.py` reproduces the selected points.

# Next Steps:

Now you have discovered many interesting insights related to the launch sites' location using folium, in a very interactive way. Next, you will need to build a dashboard using Ploty Dash on detailed launch records.


## Authors


[Yan Luo](https://www.linkedin.com/in/yan-luo-96288783/)


### Other Contributors


Joseph Santarcangelo


## Change Log


|Date (YYYY-MM-DD)|Version|Changed By|Change Description|
|-|-|-|-|
|2021-05-26|1.0|Yan|Created the initial version|


Copyright © 2021 IBM Corporation. All rights reserved.
